# CSE464 — Module 2: Instructor Colab Manual

Execute top-to-bottom in Google Colab. Each problem is self-contained and does not require a local import.

**Trace convention:** zero-based indices, unless explicitly stated. Source tags distinguish notes-based challenges from general course implementations.


## 01. Array Address Calculation

**Source classification:** Section 3

**Algorithm:** Compute offsets in row- and column-major order.

**Implementation logic:** Apply lower bounds and element width.

**Example input / execution:** `print(address_2d(1000,4,5,6,4,5,1,1))`

**Expected output:** `1088`

**Dry-run trace:** offset=(4-1)*6+(5-1)=22; address=1000+4*22=1088

**Complexity exercise:** Derive the worst-case time and auxiliary space from the implementation; identify edge cases.


In [ ]:
"""CSE464 | Array Address Calculation.
Algorithm: Compute offsets in row- and column-major order.
Logic: Apply lower bounds and element width.
"""

def address_2d(base,width,rows,cols,i,j,lr=0,lc=0,order='row'):
    if not(lr<=i<lr+rows and lc<=j<lc+cols):raise IndexError()
    offset=(i-lr)*cols+(j-lc) if order=='row' else (j-lc)*rows+(i-lr)
    return base+width*offset


print(address_2d(1000,4,5,6,4,5,1,1))


## 02. Array Stack

**Source classification:** Stack ADT

**Algorithm:** Fixed-capacity LIFO ADT.

**Implementation logic:** top is an integer and overflow/underflow are explicit.

**Example input / execution:** `s=Stack(2);s.push(10);s.push(20);print(s.pop(),s.peek())`

**Expected output:** `20 10`

**Dry-run trace:** top -1 -> 0 -> 1 -> 0

**Complexity exercise:** Derive the worst-case time and auxiliary space from the implementation; identify edge cases.


In [ ]:
"""CSE464 | Array Stack.
Algorithm: Fixed-capacity LIFO ADT.
Logic: top is an integer and overflow/underflow are explicit.
"""

class Stack:
    def __init__(self,capacity):self.a=[None]*capacity;self.top=-1
    def push(self,x):
        if self.top+1==len(self.a):raise OverflowError('full')
        self.top+=1;self.a[self.top]=x
    def pop(self):
        if self.top<0:raise IndexError('empty')
        x=self.a[self.top];self.a[self.top]=None;self.top-=1;return x
    def peek(self):
        if self.top<0:raise IndexError('empty')
        return self.a[self.top]


s=Stack(2);s.push(10);s.push(20);print(s.pop(),s.peek())


## 03. Balanced Brackets

**Source classification:** Stack applications

**Algorithm:** Match closing symbols against latest unmatched opener.

**Implementation logic:** Stack-based scan.

**Example input / execution:** `print(balanced('{[()]}'),balanced('([)]'))`

**Expected output:** `True False`

**Dry-run trace:** push { [ (; pop (, [, {

**Complexity exercise:** Derive the worst-case time and auxiliary space from the implementation; identify edge cases.


In [ ]:
"""CSE464 | Balanced Brackets.
Algorithm: Match closing symbols against latest unmatched opener.
Logic: Stack-based scan.
"""

def balanced(s):
    match={')':'(',']':'[','}':'{'};stack=[]
    for c in s:
        if c in '([{':stack.append(c)
        elif c in match:
            if not stack or stack.pop()!=match[c]:return False
    return not stack


print(balanced('{[()]}'),balanced('([)]'))


## 04. Infix To Postfix

**Source classification:** Stack expressions

**Algorithm:** Use operator stack, respecting precedence and right-associative exponentiation.

**Implementation logic:** Single-letter/digit operands; no unary operators.

**Example input / execution:** `print(postfix('A*(B+C)'))`

**Expected output:** `ABC+*`

**Dry-run trace:** A out; * stack; (; B out; + stack; C out; ) emit +; end emit *

**Complexity exercise:** Derive the worst-case time and auxiliary space from the implementation; identify edge cases.


In [ ]:
"""CSE464 | Infix To Postfix.
Algorithm: Use operator stack, respecting precedence and right-associative exponentiation.
Logic: Single-letter/digit operands; no unary operators.
"""

def postfix(expr):
    prec={'+':1,'-':1,'*':2,'/':2,'^':3};stack=[];out=[]
    for ch in expr.replace(' ',''):
        if ch.isalnum():out.append(ch)
        elif ch=='(':stack.append(ch)
        elif ch==')':
            while stack and stack[-1]!='(':out.append(stack.pop())
            if not stack:raise ValueError('mismatched parentheses')
            stack.pop()
        elif ch in prec:
            while stack and stack[-1]!='(' and (prec[stack[-1]]>prec[ch] or (prec[stack[-1]]==prec[ch] and ch!='^')):out.append(stack.pop())
            stack.append(ch)
        else:raise ValueError('invalid character')
    while stack:
        x=stack.pop()
        if x=='(':raise ValueError('mismatched parentheses')
        out.append(x)
    return ''.join(out)


print(postfix('A*(B+C)'))


## 05. Postfix Evaluation

**Source classification:** Stack expressions

**Algorithm:** Read postfix tokens left-to-right and evaluate from operand stack.

**Implementation logic:** Space-separated numeric tokens; operator pops right then left.

**Example input / execution:** `print(evaluate('2 3 4 * +'))`

**Expected output:** `14.0`

**Dry-run trace:** [2], [2,3], [2,3,4], [2,12], [14]

**Complexity exercise:** Derive the worst-case time and auxiliary space from the implementation; identify edge cases.


In [ ]:
"""CSE464 | Postfix Evaluation.
Algorithm: Read postfix tokens left-to-right and evaluate from operand stack.
Logic: Space-separated numeric tokens; operator pops right then left.
"""

def evaluate(expr):
    s=[]
    for x in expr.split():
        if x not in ('+','-','*','/','^'):s.append(float(x));continue
        b=s.pop();a=s.pop();s.append({'+':lambda:a+b,'-':lambda:a-b,'*':lambda:a*b,'/':lambda:a/b,'^':lambda:a**b}[x]())
    if len(s)!=1:raise ValueError('invalid expression')
    return s[0]


print(evaluate('2 3 4 * +'))


## 06. Prefix Evaluation

**Source classification:** Stack expressions

**Algorithm:** Scan prefix tokens in reverse with an operand stack.

**Implementation logic:** Space-separated numeric tokens.

**Example input / execution:** `print(evaluate('+ 2 * 3 4'))`

**Expected output:** `14.0`

**Dry-run trace:** reverse:4,3,*,2,+

**Complexity exercise:** Derive the worst-case time and auxiliary space from the implementation; identify edge cases.


In [ ]:
"""CSE464 | Prefix Evaluation.
Algorithm: Scan prefix tokens in reverse with an operand stack.
Logic: Space-separated numeric tokens.
"""

def evaluate(expr):
    s=[]
    for x in reversed(expr.split()):
        if x not in ('+','-','*','/','^'):s.append(float(x));continue
        a=s.pop();b=s.pop();s.append({'+':lambda:a+b,'-':lambda:a-b,'*':lambda:a*b,'/':lambda:a/b,'^':lambda:a**b}[x]())
    if len(s)!=1:raise ValueError('invalid expression')
    return s[0]


print(evaluate('+ 2 * 3 4'))


## 07. Linear Queue

**Source classification:** Queue ADT

**Algorithm:** Array FIFO with front and rear indices.

**Implementation logic:** Show false overflow after removals.

**Example input / execution:** `q=Queue(3);q.enqueue(5);q.enqueue(6);print(q.dequeue(),q.dequeue())`

**Expected output:** `5 6`

**Dry-run trace:** front=0,rear=1 -> front=2,rear=1

**Complexity exercise:** Derive the worst-case time and auxiliary space from the implementation; identify edge cases.


In [ ]:
"""CSE464 | Linear Queue.
Algorithm: Array FIFO with front and rear indices.
Logic: Show false overflow after removals.
"""

class Queue:
    def __init__(self,n):self.a=[None]*n;self.front=0;self.rear=-1
    def enqueue(self,x):
        if self.rear+1==len(self.a):raise OverflowError('rear exhausted')
        self.rear+=1;self.a[self.rear]=x
    def dequeue(self):
        if self.front>self.rear:raise IndexError('empty')
        x=self.a[self.front];self.front+=1;return x


q=Queue(3);q.enqueue(5);q.enqueue(6);print(q.dequeue(),q.dequeue())


## 08. Circular Queue

**Source classification:** Circular queue

**Algorithm:** Wrap indices modulo capacity to reuse array slots.

**Implementation logic:** Track size to distinguish empty/full.

**Example input / execution:** `q=CircularQueue(2);q.enqueue(1);q.enqueue(2);print(q.dequeue());q.enqueue(3);print(q.dequeue(),q.dequeue())`

**Expected output:** `1 / 2 3`

**Dry-run trace:** front 0 -> 1 -> 0; slot 0 reused

**Complexity exercise:** Derive the worst-case time and auxiliary space from the implementation; identify edge cases.


In [ ]:
"""CSE464 | Circular Queue.
Algorithm: Wrap indices modulo capacity to reuse array slots.
Logic: Track size to distinguish empty/full.
"""

class CircularQueue:
    def __init__(self,n):
        if n<=0:raise ValueError('capacity must be positive')
        self.a=[None]*n;self.front=0;self.count=0
    def enqueue(self,x):
        if self.count==len(self.a):raise OverflowError('full')
        self.a[(self.front+self.count)%len(self.a)]=x;self.count+=1
    def dequeue(self):
        if not self.count:raise IndexError('empty')
        x=self.a[self.front];self.a[self.front]=None;self.front=(self.front+1)%len(self.a);self.count-=1;return x


q=CircularQueue(2);q.enqueue(1);q.enqueue(2);print(q.dequeue());q.enqueue(3);print(q.dequeue(),q.dequeue())


## 09. Priority Queue

**Source classification:** Priority queue

**Algorithm:** Store priority-labelled items in a binary min-heap.

**Implementation logic:** Lower numeric priority is served first; sequence breaks ties.

**Example input / execution:** `q=PriorityQueue();q.enqueue(3,'low');q.enqueue(1,'high');q.enqueue(1,'high2');print([q.dequeue() for _ in range(3)])`

**Expected output:** `['high', 'high2', 'low']`

**Dry-run trace:** priority 1 FIFO tie order, then priority 3

**Complexity exercise:** Derive the worst-case time and auxiliary space from the implementation; identify edge cases.


In [ ]:
"""CSE464 | Priority Queue.
Algorithm: Store priority-labelled items in a binary min-heap.
Logic: Lower numeric priority is served first; sequence breaks ties.
"""

import heapq
class PriorityQueue:
    def __init__(self):self.heap=[];self.seq=0
    def enqueue(self,p,x):heapq.heappush(self.heap,(p,self.seq,x));self.seq+=1
    def dequeue(self):
        if not self.heap:raise IndexError('empty')
        return heapq.heappop(self.heap)[2]


q=PriorityQueue();q.enqueue(3,'low');q.enqueue(1,'high');q.enqueue(1,'high2');print([q.dequeue() for _ in range(3)])


## 10. Infix To Prefix

**Source classification:** Stack expressions

**Algorithm:** Reverse the postfix reasoning via recursive parsing of precedence.

**Implementation logic:** Single-letter operands and parentheses, exponent is right-associative.

**Example input / execution:** `print(prefix('A*(B+C)'))`

**Expected output:** `* A + B C`

**Dry-run trace:** parse multiply with right subtree plus

**Complexity exercise:** Derive the worst-case time and auxiliary space from the implementation; identify edge cases.


In [ ]:
"""CSE464 | Infix To Prefix.
Algorithm: Reverse the postfix reasoning via recursive parsing of precedence.
Logic: Single-letter operands and parentheses, exponent is right-associative.
"""

def prefix(expr):
    s=expr.replace(' ','');i=0
    def parse(minprec=0):
        nonlocal i
        if i>=len(s):raise ValueError('missing operand')
        if s[i]=='(':
            i+=1;left=parse()
            if i>=len(s) or s[i]!=')':raise ValueError('mismatched')
            i+=1
        else:
            left=s[i];i+=1
            if not left.isalnum():raise ValueError('invalid operand')
        prec={'+':1,'-':1,'*':2,'/':2,'^':3}
        while i<len(s) and s[i] in prec and prec[s[i]]>=minprec:
            op=s[i];i+=1;right=parse(prec[op]+(0 if op=='^' else 1));left=op+' '+left+' '+right
        return left
    result=parse()
    if i!=len(s):raise ValueError('invalid expression')
    return result


print(prefix('A*(B+C)'))
